In [22]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from lime.lime_tabular import LimeTabularExplainer
from tqdm import tqdm

ind_hosp = pd.read_parquet("ind_hosp.parquet")
calibrated = joblib.load("lightgbm.pkl")
base_model = calibrated.calibrated_classifiers_[0].estimator

cols_to_drop = [
    "subject_id",
    "hadm_id",
    "dischtime",
    "current_date",
    "target_readmission_30d",
    "los"
]

X = ind_hosp.drop(columns=cols_to_drop)
y = ind_hosp["target_readmission_30d"]

bool_cols = X.select_dtypes(include="bool").columns
X[bool_cols] = X[bool_cols].astype(int)

patient_target = ind_hosp.groupby('subject_id')['target_readmission_30d'].max().reset_index()
patient_target.columns = ['subject_id', 'has_readmission']

train_val_ids, test_ids = train_test_split(
    patient_target['subject_id'],
    test_size=0.2,
    random_state=42,
    stratify=patient_target['has_readmission']
)

train_ids, val_ids = train_test_split(
    train_val_ids,
    test_size=0.125,
    random_state=42,
    stratify=patient_target[patient_target['subject_id'].isin(train_val_ids)]['has_readmission']
)

train_mask = ind_hosp['subject_id'].isin(train_ids)
val_mask = ind_hosp['subject_id'].isin(val_ids)
test_mask = ind_hosp['subject_id'].isin(test_ids)

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

In [23]:
print(f"Rows: {len(X_test)}")
print(f"Hospitalizations: {ind_hosp.loc[test_mask, 'hadm_id'].nunique()}")
print(f"Patients: {ind_hosp.loc[test_mask, 'subject_id'].nunique()}")

Rows: 365442
Hospitalizations: 81616
Patients: 35915


In [24]:
X_train_lime = X_train.copy()
X_test_lime = X_test.copy()

icd_cols = [c for c in X_train.columns if c.startswith("icd_")]
ccsr_cols = [c for c in X_train.columns if c.startswith("ccsr_")]

X_train_lime[icd_cols] = X_train_lime[icd_cols].fillna(0)
X_test_lime[icd_cols] = X_test_lime[icd_cols].fillna(0)

In [25]:
clinical_ranges = {
    'lab_50983_daily': {'low': 135, 'high': 145}, #sodium
    'lab_50971_daily': {'low': 3.5, 'high': 5}, #potassium
    'lab_50902_daily': {'low': 95, 'high': 105}, #chloride
    'lab_50882_daily': {'low': 22, 'high': 32}, #bicarbonate
    'lab_50912_daily': {'low': 0.8, 'high': 1.3}, #creatinine
    'lab_51006_daily': {'low': 8, 'high': 21}, #BUN
    'lab_50931_daily': {'low': 65, 'high': 110}, #glucose
    'lab_50893_daily': {'low': 8.5, 'high': 10.2}, #calcium
    'lab_50868_daily': {'low': 10, 'high': 18}, #anion gap
    'lab_51222_daily': {
        'male': {'low': 13, 'high': 17},
        'female': {'low': 12, 'high': 15}
    }, #hemoglobin
    'lab_51301_daily': {'low': 4, 'high': 10}, #WBC
    'lab_51265_daily': {'low': 150, 'high': 400}, #Platelet Count
    'lab_51221_daily': {
        'male': {'low': 40, 'high': 52},
        'female': {'low': 36, 'high': 47}
    }, #Hematocrit
    'lab_51250_daily': {'low': 80, 'high': 100}, #MCV
    'lab_51277_daily': {'low': 11.5, 'high': 14.5}, #RDW
    'lab_50960_daily': {'low': 1.5, 'high': 2}, #magnesium
    'lab_50970_daily': {'low': 2.7, 'high': 4.5}, #phosphate
    'lab_51248_daily': {'low': 26, 'high': 32}, #MCH
    'lab_51249_daily': {'low': 30, 'high': 35}, #MCHC
    'lab_51279_daily': {
        'male': {'low': 4.5, 'high': 5.9},
        'female': {'low': 4, 'high': 5.2}
    } #RBC
}

def normal_lab_value(col, gender=None):
    ref = clinical_ranges[col]
    if "male" in ref:
        if gender == 1:
            low = ref["male"]["low"]
            high = ref["male"]["high"]
        else:
            low = ref["female"]["low"]
            high = ref["female"]["high"]
    else:
        low = ref["low"]
        high = ref["high"]

    return (low + high) / 2

In [26]:
lab_cols = [c for c in X_train.columns if c.startswith("lab_")]

for col in lab_cols:
    if col not in clinical_ranges:
        continue

    missing = X_train_lime[col].isna()
    male_idx = missing & (X_train_lime["gender_male"] == 1)
    female_idx = missing & (X_train_lime["gender_male"] == 0)
    X_train_lime.loc[male_idx, col] = normal_lab_value(col, 1)
    X_train_lime.loc[female_idx, col] = normal_lab_value(col, 0)

for col in lab_cols:
    if col not in clinical_ranges:
        continue

    missing = X_test_lime[col].isna()
    male_idx = missing & (X_test_lime["gender_male"] == 1)
    female_idx = missing & (X_test_lime["gender_male"] == 0)
    X_test_lime.loc[male_idx, col] = normal_lab_value(col, 1)
    X_test_lime.loc[female_idx, col] = normal_lab_value(col, 0)

In [27]:
#X_test_lime = X_test_lime[:100]
icd_cols = [col for col in X.columns if col.startswith('icd_')]
ccsr_cols = [col for col in X.columns if col.startswith('ccsr_')]
lab_cols = [col for col in X.columns if col.startswith('lab_') and col.endswith('_daily')]

features_to_analyze = (
    icd_cols +
    ccsr_cols +
    lab_cols +
    [
        'num_diagnoses',
        'num_chronic',
        'comorbidity_score',
        'num_medications_daily',
        'prior_admissions_12mo',
        'cumulative_procedures',
        'cumulative_medications',
        'num_procedures_daily',
        'gender_male',
        'age'
    ]
)

explainer = LimeTabularExplainer(
    training_data=X_train_lime.values,
    feature_names=X_train_lime.columns.tolist(),
    class_names=["No readmission", "Readmission"],
    mode="classification",
    discretize_continuous=False,
    random_state=42
)

predict_fn = lambda x: calibrated.predict_proba(pd.DataFrame(x, columns=X_train_lime.columns))
all_results = []
for idx in tqdm(X_test_lime.index,
                total=len(X_test_lime),
                desc="LIME"):

    row = X_test_lime.loc[idx]
    exp = explainer.explain_instance(
        row.values,
        predict_fn,
        num_features=len(X_train_lime.columns),
        num_samples=1000
    )

    weights = {
        X_train_lime.columns[i]: w
        for i, w in exp.local_exp[1]
    }

    pred = predict_fn(row.values.reshape(1, -1))[0, 1]
    for feature in features_to_analyze:

        all_results.append({
            "subject_id": ind_hosp.loc[idx, "subject_id"],
            "hadm_id": ind_hosp.loc[idx, "hadm_id"],
            "feature": feature,
            "lime": weights.get(feature, 0.0),
            "value": row[feature],
            "predicted_proba": pred,
        })

lime_df = pd.DataFrame(all_results)

LIME: 100%|██████████| 365442/365442 [1:25:49<00:00, 70.97it/s]


In [28]:
lime_by_hadm = (
    lime_df
    .groupby(["subject_id", "hadm_id", "feature"])
    .agg(
        lime_mean=("lime", "mean"),
        lime_mean_abs=("lime", lambda x: np.mean(np.abs(x))),
        value_mean=("value", "mean"),
        predicted_proba_mean=("predicted_proba", "mean")
    )
    .reset_index()
)

lime_summary = (
    lime_by_hadm
    .groupby("feature")
    .agg(
        lime_mean=("lime_mean", "mean"),
        lime_mean_abs=("lime_mean_abs", "mean"),
        value_mean=("value_mean", "mean")
    )
    .reset_index()
)

In [29]:
lime_by_hadm.to_csv("lime_by_hadm.csv", index=False)
lime_summary.to_csv("lime_summary.csv", index=False)

### Results

In [3]:
lab_names = {
    '50983': 'Sodium',
    '50971': 'Potassium',
    '50902': 'Chloride',
    '50882': 'Bicarbonate',
    '50912': 'Creatinine',
    '51006': 'BUN',
    '50931': 'Glucose',
    '50893': 'Calcium',
    '50868': 'Anion Gap',
    '51222': 'Hemoglobin',
    '51301': 'WBC',
    '51265': 'Platelet Count',
    '51221': 'Hematocrit',
    '51250': 'MCV',
    '51277': 'RDW',
    '50960': 'Magnesium',
    '50970': 'Phosphate',
    '51248': 'MCH',
    '51249': 'MCHC',
    '51279': 'RBC'
}

icd_names = {
    'I10': 'Essential (primary) hypertension',
    'E785': 'Hyperlipidemia, unspecified',
    'K219': 'Gastroesophageal reflux disease without esophagitis',
    'Z87891': 'Personal history of nicotine dependence',
    'I2510': 'Atherosclerotic heart disease of native coronary artery without angina pectoris',
    'N179': 'Acute kidney failure, unspecified',
    'F329': 'Major depressive disorder, single episode, unspecified',
    'I4891': 'Unspecified atrial fibrillation',
    'Z7901': 'Long term (current) use of anticoagulants',
    'F419': 'Anxiety disorder, unspecified',
    'E119': 'Type 2 diabetes mellitus without complications',
    'E039': 'Hypothyroidism, unspecified',
    'Z794': 'Long term (current) use of insulin',
    'D649': 'Anemia, unspecified',
    'N390': 'Urinary tract infection, site not specified'
}

ccsr_names = {
    'FAC021': 'Personal/family history of disease',
    'FAC025': 'Other specified status',
    'END011': 'Fluid and electrolyte disorders',
    'CIR011': 'Coronary atherosclerosis and other heart disease',
    'END010': 'Disorders of lipid metabolism',
    'CIR007': 'Essential hypertension',
    'END003': 'Diabetes mellitus with complication',
    'CIR019': 'Heart failure',
    'DIG004': 'Esophageal disorders',
    'CIR017': 'Cardiac dysrhythmias',
    'CIR008': 'Hypertension with complications and secondary hypertension',
    'BLD003': 'Aplastic anemia',
    'EXT027': 'External cause codes: place of occurrence of the external cause',
    'GEN002': 'Acute and unspecified renal failure',
    'END009': 'Obesity'
}

def map_feature_name(feature_name):
    if feature_name.startswith('lab_') and feature_name.endswith('_daily'):
        code = feature_name.replace('lab_', '').replace('_daily', '')
        if code in lab_names:
            return f"{lab_names[code]} ({code})"
    elif feature_name.startswith('icd_'):
        code = feature_name.replace('icd_', '')
        if code in icd_names:
            return f"{icd_names[code]} ({code})"
    elif feature_name.startswith('ccsr_'):
        code = feature_name.replace('ccsr_', '')
        if code in ccsr_names:
            return f"{ccsr_names[code]} ({code})"
    return feature_name

In [15]:
import pandas as pd
lime_summary = pd.read_csv('lime_summary.csv')
lime_summary['feature'] = lime_summary['feature'].apply(map_feature_name)
lime_summary = lime_summary.sort_values(by="lime_mean_abs", ascending=False)
lime_summary

,feature,lime_mean,lime_mean_abs,value_mean
59,prior_admissions_12mo,0.033289,0.033289,1.206099
16,comorbidity_score,-0.015359,0.015359,1.743004
6,Heart failure (CIR019),-0.004961,0.005105,0.167100
0,age,-0.004819,0.004965,60.273047
51,Platelet Count (51265),0.004081,0.004360,247.893765
49,MCHC (51249),0.003221,0.003708,32.657231
48,MCH (51248),0.002964,0.003516,29.403820
8,Diabetes mellitus with complication (END003),-0.002560,0.003276,0.125110
32,Long term (current) use of anticoagulants (Z7901),-0.002514,0.003243,0.114144
22,Type 2 diabetes mellitus without complications...,0.002426,0.003188,0.128051
